## initialize

In [1]:
# init
from importlib import reload
import os
from pathlib import Path
import pandas as pd
from IPython.display import clear_output
import boto3
import sys
import time

import toolsGeneral.main as tgm
import toolsGeneral.logger as tgl
import toolsOSM.overpass as too
import toolsPandas.helpers as tph
import toolsSync.main as tsm

def pckgs_reload():
    reload(tgm)
    reload(too)
    reload(tph)
    reload(tgl)
    reload(tsm)

ROOT = Path.cwd().parents[0]
DATA_DIR = ROOT / "data"
TESTS_DIR = DATA_DIR / 'tests results'
CLEANED_DIR = DATA_DIR / 'cleaned'
DEV_MODE = True

task = 'test_first_level'
TEST_FIRST_LEVEL_DIR = TESTS_DIR / 'osm first level test'
process_state = tgm.load(DATA_DIR / "process_state.json")
first_level_test_state = tgm.load(DATA_DIR / "first_level_test_state.json")

logger = tgl.initiate_logger('logger', TEST_FIRST_LEVEL_DIR / 'first_level_test.log')

## osm test if first level divisions belong to country

### * select countries to test

In [2]:
countries_tested = [c for c, val in process_state.items() if (val[task]['status'] == 'ok')]
logger.info(f"countries tested: {len(countries_tested)}")
countries_to_test = [c for c, val in process_state.items() if 
    (val['clean']['status'] == 'ok') and (val[task]['status'] in ['pending', 'error'])
]
logger.info(f"countries to test: {len(countries_to_test)}")

[INFO] : countries tested: 83
[INFO] : countries to test: 0


In [3]:
countries_to_test = ['Andorra',
 'VaticanCity',
 'FaroeIslands',
 'NorthMacedonia',
 'Montenegro',
 'SanMarino',
 'IsleOfMan',
 'Ireland',
 'Latvia',
 'Estonia',
 'Azerbaijan',
 'Bahamas',
 'Barbados',
 'Bermuda',
 'Anguilla']

### * initialize B2

In [4]:
session = boto3.session.Session()

s3 = session.client(
    service_name="s3",
    aws_access_key_id=os.environ["B2_KEY_ID"],
    aws_secret_access_key=os.environ["B2_APPLICATION_KEY"],
    endpoint_url=os.environ["B2_ENDPOINT"]
)

### * download required data

In [5]:
logger.info(f"* Downloading required data to test: {len(countries_to_test)} countries")
logger.info(f"  * Downloading data from B2 in directory: '{CLEANED_DIR.relative_to(ROOT)}'")

countries_downloaded = tsm.donwload_country_data_from_bucket(countries_to_test, os.environ["B2_BUCKET_NAME"], CLEANED_DIR.relative_to(ROOT), CLEANED_DIR, s3, logger)

logger.info(f'* Countries to test: {len(countries_to_test)}')
logger.info(f"* Countries to test with downloaded cleaned data from B2: {len(countries_downloaded)}")

[INFO] : * Downloading required data to test: 1 countries
[INFO] :   * Downloading data from B2 in directory: 'data\cleaned'
[INFO] :   * Countries to get data: 1
[INFO] :     * Found in B2: 1, Missing in B2: 0
[INFO] :   * Downloading data for countries: 1
[INFO] :     * Directory: 'data\cleaned' -> 'c:\Users\gonta\D\study\full stack\projects\administrative divisions osm scrape\data\cleaned'
[INFO] :     * (1/1) Country Armenia files found: 1
[INFO] :       * Skip existing file Armenia_cleaned.pkl
[INFO] :   * Number of downloaded files: 1/1
[INFO] : * Countries to test: 1
[INFO] : * Countries to test with downloaded cleaned data from B2: 1


### * load data for countries to test

In [4]:
logger.info(f"* Load data from: {CLEANED_DIR.relative_to(ROOT)}")
cleaned_files = [f for f in CLEANED_DIR.glob('*/*')]
logger.info(f"  * Directories found: {len(cleaned_files)}")

# to_test_df = {file.parent.name:tgm.load(str(file)) for file in cleaned_files if file.parent.name}
countries_to_test_df = {file.parent.name:tgm.load(str(file)) for file in cleaned_files if file.parent.name in countries_to_test}
logger.info(f"  * Countries to test: {len(countries_to_test)}")
logger.info(f"  * Data loaded for countries to test: {len(countries_to_test_df)}")

[INFO] : * Load data from: data\cleaned
[INFO] :   * Directories found: 83
[INFO] :   * Countries to test: 15
[INFO] :   * Data loaded for countries to test: 15


In [5]:
first_lvl_df = {}
countries_wihout_first_level = []
for country, df in countries_to_test_df.items():
    if not df[df['tags.admin_level'] == '4'].empty:
        first_lvl_df[country] = df[df['tags.admin_level'] == '4']
    else:
        countries_wihout_first_level.append(country)
logger.info(f"countries with first level: {len(first_lvl_df)}")
logger.info(f"countries without first level: {countries_wihout_first_level}")

[INFO] : countries with first level: 0
[INFO] : countries without first level: ['Andorra', 'Anguilla', 'Azerbaijan', 'Bahamas', 'Barbados', 'Bermuda', 'Estonia', 'FaroeIslands', 'Ireland', 'IsleOfMan', 'Latvia', 'Montenegro', 'NorthMacedonia', 'SanMarino', 'VaticanCity']


### * filter already processed and failed relations

In [ ]:
logger.info(f"First_level_test_state countries: {len(first_level_test_state)}")
logger.info(f"First_level_test_state triplets processed: {sum([len(t['processed']) for t in first_level_test_state.values()])}")

[INFO] : first_level_test_state: 61
[INFO] : tuples processed: 1202


In [ ]:
first_lvl_filtered_df = {}
for country,df in first_lvl_df.items():
    processed = first_level_test_state[country]['processed'] if first_level_test_state.get(country) else set()
    failed = first_level_test_state[country]['failed'] if first_level_test_state.get(country) else set()

    in_processed = df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(processed)
    in_failed = df[['id','tags.parent_id','tags.country_id']].apply(tuple, axis=1).isin(failed)
    filtered_df = df[~in_processed | in_failed]
    if not filtered_df.empty:
        first_lvl_filtered_df[country] = filtered_df

logger.info(f"Countries with first level -> filtered pending to process: {len(first_lvl_filtered_df)} \n {list(first_lvl_filtered_df.keys())}")
logger.info(f"Total of relations to process: {sum([len(lis) for lis in first_lvl_filtered_df.values()])}")

[INFO] : Countries with first level -> filtered pending to process: 1
[INFO] : Total of pending to process: 9


### * make tests

In [ ]:
for country, df in first_lvl_filtered_df.items():
    logger.info(f"* Testing country {country}: {len(df)} relations")

    if not first_level_test_state.get(country):
        first_level_test_state[country] = {"to_process": set(), "processed": set(), "failed": set(), "next_index": 0}
    country_test_state = first_level_test_state[country]

    total = len(df)
    test_res = {}
    
    CHUNK_SIZE = 15
    MAX_SECONDS_WITHOUT_UPLOAD = 300 # 5 minutes
    chunk_count = 0
    last_upload_time = time.time()
    save_path = TEST_FIRST_LEVEL_DIR / country / f"{country}_first_level_test_res_{country_test_state['next_index']}.pkl"

    for i, (idx, row) in enumerate(df.iterrows(), start=1):
        id_triplet = (row["id"], row["tags.parent_id"], row["tags.country_id"])
        logger.info(f" ^ [{i}/{total}]: testing {id_triplet}:")

        res = too.is_child_inside_parent(row["id"], row["tags.parent_id"])

        test_res[id_triplet] = res

        status_list = [v['status'] for k,v in res.items()]
        if 'error' in status_list:
            country_test_state['failed'].add(id_triplet)
        else:
            country_test_state['processed'].add(id_triplet)
            country_test_state['failed'].discard(id_triplet)

        time.sleep(2)
        resume  = {k:v['status'] for k,v in res.items()}
        logger.info(f" $ finished: status: {resume}")

        chunk_count += 1
        #* persist partial results
        if (chunk_count >= CHUNK_SIZE or (time.time() - last_upload_time) >= MAX_SECONDS_WITHOUT_UPLOAD):
            logger.info(f"* Checkpoint upload for {country}")
            # persist current chunk
            tgm.dump(save_path, test_res)
            # advance chunk
            test_res = {}
            country_test_state['next_index'] += 1
            # next chunk file
            save_path = TEST_FIRST_LEVEL_DIR / country / f"{country}_first_level_test_res_{country_test_state['next_index']}.pkl"

            tgm.dump(DATA_DIR / "first_level_test_state.json", first_level_test_state)
            # upload to B2
            if not DEV_MODE:
                logger.info("* Uploading data to backblaze b2")
                tsm.upload_dir_files_to_backblaze(TEST_FIRST_LEVEL_DIR / country, config)
                tsm.commit_file(DATA_DIR  / "first_level_test_state.json", f"Update {country} first level test state: chunk {country_test_state['next_index'] - 1}", logger)

            chunk_count = 0
            last_upload_time = time.time()


    logger.info(f"* Finished {country}: saving data ...")
    # save test res
    if test_res:
        tgm.dump(save_path, test_res)
        country_test_state['next_index'] += 1

    # save country state
    tgm.dump(DATA_DIR / "first_level_test_state.json", first_level_test_state)

    if len(first_level_test_state[country]['failed']) < 1:
        tsm.update_process_state(process_state, country, task, process_status='ok')
        tgm.dump(DATA_DIR / "process_state.json", process_state)

    # upload and commit after a country finishes
    if not DEV_MODE:
        logger.info("* Uploading data to backblaze b2")
        tsm.upload_dir_files_to_backblaze(TEST_FIRST_LEVEL_DIR / country, config)
        tsm.commit_file(DATA_DIR  / "first_level_test_state.json", f"Update {country} first level test state", logger)
        tsm.commit_file(DATA_DIR / "process_state.json", f"Update process state for {country}: ({task}, ok)", logger)

[INFO] :  ^ [1/3]: testing ('364077', '364066', '364066'):
[INFO] :    > Getting admin_centre:
[INFO] :     * Found node (admin_centre)
[INFO] :    > Testing admin_centre node (id: 130037434)
[INFO] :     * Finished testing (admin_centre): True
[INFO] :    > Getting label:
[INFO] :     * Missing node (label)
[INFO] :    > Getting place:
[INFO] :     * Found node (place)
[INFO] :    > Testing place node (id: 130037434)
[INFO] :    * Attempt 1 failed: http_error: 429
[INFO] :     * Finished testing (place): True
[INFO] :    > Getting geom_node:
[INFO] :    * Attempt 1 failed: http_error: 504
[INFO] :    * Attempt 2 failed: http_error: 504
[INFO] :     * Found node (geom_node)
[INFO] :    > Testing geom_node node (id: 150574770)
[INFO] :     * Finished testing (geom_node): True
[INFO] :    > Getting centroid:
[INFO] :     * Found node (centroid)
[INFO] :    > Testing centroid (lat: 40.805829, lon: 43.8215162)
[INFO] :    * Attempt 1 failed: http_error: 504
[INFO] :     * Finished testing 

### * [dev] load test res data

In [ ]:
test_res_by_cntr = tgm.load_countries_dirs(TEST_FIRST_LEVEL_DIR, countries_to_test)
logger.info(f'Results per country found: {len(test_res_by_cntr)}')

test_res_by_tuple = {k:v for list in test_res_by_cntr.values() for k,v in list.items()}
logger.info(f'Results per element found: {len(test_res_by_tuple)}')

[INFO] : Results per country found: 1
[INFO] : Results per element found: 11


### * [dev] review

In [37]:
result_status = [[[k,v['status']] for k,v in ele.items()] for ele in test_res_by_tuple.values()]
pd.Series(list(result_status)).value_counts()

[[admin_centre, ok], [label, missing], [place, ok], [geom_node, ok], [centroid, ok]]    10
[[admin_centre, ok], [label, ok], [place, ok], [geom_node, ok], [centroid, ok]]          1
Name: count, dtype: int64

In [38]:
pd.Series([ele[1] for lis in result_status for ele in lis]).value_counts()

ok         45
missing    10
Name: count, dtype: int64

In [39]:
results = [[[k,v['result']] if v['status'] == 'ok' else  [k,v['status']] for k,v in ele.items() ] for ele in test_res_by_tuple.values()]
pd.Series(results).value_counts()

[[admin_centre, True], [label, missing], [place, True], [geom_node, True], [centroid, True]]    10
[[admin_centre, True], [label, True], [place, True], [geom_node, True], [centroid, True]]        1
Name: count, dtype: int64